# Compare predictions slice by slice (3D Slicer style)

Pick an image (and its ground truth), tick the runs to compare and click **Predict and show**. Each run predicts the image with its own preprocessing (z-score or raw intensities), as `hippo.visualize` does. You then get **one row per mask**, ground truth first and then each run, and each row has the axial, coronal and sagittal views. The three sliders at the top move the same slice in **every row**, like linked views in 3D Slicer, so you compare the masks at exactly the same place.

| View | Screen left / right | Screen top / bottom | Colour |
|---|---|---|---|
| **Axial** | R / L | A / P | red |
| **Coronal** | R / L | S / I | green |
| **Sagittal** | A / P | S / I | yellow |

**Colour** (for the run rows):
- **Labels**: the prediction itself. Orange = anterior, blue = posterior.
- **Errors**: each voxel compared with the ground truth. Green = correct, yellow = wrong part (anterior vs posterior), blue = missed (in the ground truth, not predicted), red = extra (predicted, not in the ground truth). For the external dataset, whose labels are subfields, only the whole hippocampus is compared, so there is no yellow.

The ground truth row keeps its own colours: anterior/posterior (MSD), or gold = CA1-3, green = subiculum, purple = CA4-DG (external, names inferred from the atlas). Each run's row title gives its Dice against the ground truth.

**Which files are listed:** only hippocampus crops, because the model was trained on crops:
- MSD images: `data/Task04_Hippocampus/imagesTr`, `imagesTs` (no ground truth)
- external crops: `data/external/imagesTr`, made by `python -m hippo.external --src data/mri_dataset --run runs/baseline`. Run it again with more `--subjects`, or without `--subjects` for all 25, to get more crops here.

The whole-brain scans in `data/mri_dataset` are not listed; open them in `view_slices.ipynb`. Predictions are also saved in `runs/figures/notebook/` (`<case>_pred_<run>.nii.gz`), so `view_slices.ipynb` can open them.

Run all cells, then use the controls under the last cell. Research use only, not for diagnosis.

In [ ]:
# Work from the project root (so data/ and runs/ resolve) and import hippo from src/
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hippo").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import ipywidgets as w
import numpy as np
from IPython.display import display

from hippo.slices import (ERRORS, NONE, VIEWS, Scan, error_map, guess_mask, label_colours, legend_html,
                          list_nifti, mm_per_px, on_grid, rgb_colours)

print(f"project: {ROOT}")

In [ ]:
import nibabel as nib
import yaml

from hippo import visualize

OUT = Path("runs/figures/notebook")  # predictions are saved here
OUT.mkdir(parents=True, exist_ok=True)
RUNS = sorted(str(p.parent) for p in Path("runs").glob("*/best.pt"))
MAX_CROP = 128  # longest side (voxels) accepted as a hippocampus crop
PRED_COLOURS = rgb_colours(label_colours({1, 2}))  # anterior orange, posterior blue
ERROR_COLOURS = rgb_colours(ERRORS)


def predict(run, image):
    """Save the run's prediction for `image` as NIfTI (aligned with the MRI by its affine); returns the path.

    The image is preprocessed the way the run was trained (z-score or raw intensities).
    """
    run = Path(run)
    normalize = yaml.safe_load((run / "config.yaml").read_text())["data"].get("normalize", "zscore")
    d = visualize.load(Path(image), None, normalize)
    mask = visualize.predict(run, d["image"])
    path = OUT / f"{Path(image).name.split('.nii')[0]}_pred_{run.name}.nii.gz"
    nib.save(nib.Nifti1Image(mask.astype(np.uint8), np.asarray(d["image"].affine, dtype=np.float64)), path)
    return path


def dice_text(pred, truth, parts):
    """Whole-hippocampus Dice, plus anterior/posterior when the ground truth has those labels."""
    p, t = pred > 0, truth > 0
    text = f"Dice whole {2 * (p & t).sum() / max(p.sum() + t.sum(), 1):.3f}"
    if parts:
        text += ", " + ", ".join(f"{name} {d:.3f}" for name, d in visualize.dice(pred, truth).items())
    return text

In [ ]:
# Hippocampus crops only: MSD images*/ folders and the external crops in data/external/imagesTr
# (whole-brain scans such as data/mri_dataset are left out; the model was trained on crops)
files = [f for f in list_nifti() if Path(f).parent.name.startswith("images")]
truths = [f for f in list_nifti() if Path(f).parent.name.startswith("labels")]
wide = w.Layout(width="95%")
search = w.Text(placeholder="type to filter, e.g.  hippocampus_135   or   s03_L", description="Filter", layout=wide)
image_dd = w.Dropdown(options=files, description="Image", layout=wide)
label_dd = w.Dropdown(options=[NONE] + truths, description="Truth", layout=wide)
runs_sel = w.SelectMultiple(options=RUNS, value=tuple(RUNS), description="Runs", rows=min(len(RUNS), 6))
go_btn = w.Button(description="Predict and show", button_style="primary", icon="play")
colour_tb = w.ToggleButtons(options=[("Labels", "labels"), ("Errors", "errors")], value="labels", description="Colour")
contrast = w.IntRangeSlider(value=(1, 99), min=0, max=100, description="Contrast %")
opacity = w.FloatSlider(value=0.5, min=0, max=1, step=0.05, description="Opacity")
crosshair = w.Checkbox(value=True, description="Crosshair", indent=False)
size = w.IntSlider(value=240, min=120, max=480, step=20, description="Size px")
legend = w.HTML()
info = w.HTML()
status = w.HTML()
sliders = [w.IntSlider(description=name, continuous_update=True, disabled=True, layout=w.Layout(width="32%"),
                       style={"handle_color": colour}) for name, _, _, colour in VIEWS]
positions = w.HTML()
grid = w.VBox()
# rows: [{"title", "labels", "colours", "errors", "pictures"}]
state = {"scan": None, "pos": None, "rows": [], "ready": False}


def render(_=None):
    scan, pos = state["scan"], state["pos"]
    if not state["ready"]:
        return
    scale = mm_per_px(scan.data.shape, scan.spacing, size.value)
    errors = colour_tb.value == "errors"
    for row in state["rows"]:
        labels, colours = (row["errors"], ERROR_COLOURS) if errors and row["errors"] is not None \
            else (row["labels"], row["colours"])
        for picture, view in zip(row["pictures"], VIEWS):
            picture.value = scan.render(view, pos, contrast.value, opacity.value, crosshair.value, scale,
                                        labels=labels, colours=colours)
    legend.value = legend_html(ERROR_COLOURS if errors else PRED_COLOURS)
    positions.value = "<small>" + " &nbsp; | &nbsp; ".join(
        f"{name} slice {pos[axis] + 1} / {scan.data.shape[axis]} ({'xyz'[axis]} = {scan.mm(axis, pos[axis]):.1f} mm)"
        for name, axis, _, _ in VIEWS) + "</small>"
    here = ", ".join(f"{row['title'].split(':')[0]} {int(row['labels'][tuple(pos)])}" for row in state["rows"])
    info.value = f"<code>{scan.describe(pos)}</code><br><small>label at the crosshair: {here}</small>"


def on_slide(i, change):
    if state["ready"]:
        state["pos"][VIEWS[i][1]] = change["new"]
        render()


def add_row(title, labels, colours, errors=None):
    pictures = [w.Image(format="png") for _ in VIEWS]
    state["rows"].append(dict(title=title, labels=labels, colours=colours, errors=errors, pictures=pictures))
    return w.VBox([w.HTML(f"<b>{title}</b>"),
                   w.HBox([w.Box([p], layout=w.Layout(border=f"2px solid {colour}", margin="2px"))
                           for p, (_, _, _, colour) in zip(pictures, VIEWS)])])


def on_go(_=None):
    if not image_dd.value or not runs_sel.value:
        status.value = "Pick an image and at least one run."
        return
    state["ready"] = False
    try:
        scan = Scan(image_dd.value)
        if max(scan.data.shape) > MAX_CROP:
            raise ValueError(f"{scan.data.shape} voxels looks like a whole-brain scan; the model expects a "
                             "hippocampus crop (MSD, or data/external/imagesTr)")
        truth = on_grid(label_dd.value, scan.data.shape, scan.spacing, scan.origin) if label_dd.value != NONE else None
        state["rows"] = []
        rows = []
        if truth is not None:
            values = set(np.unique(truth).tolist()) - {0}
            parts = values <= {1, 2}
            rows.append(add_row("ground truth", truth, rgb_colours(label_colours(values))))
        for run in runs_sel.value:
            status.value = f"Predicting with {Path(run).name} ..."
            pred = on_grid(predict(run, image_dd.value), scan.data.shape, scan.spacing, scan.origin)
            title = Path(run).name + (f": {dice_text(pred, truth, parts)}" if truth is not None else "")
            rows.append(add_row(title, pred, PRED_COLOURS, error_map(pred, truth, parts) if truth is not None else None))
    except Exception as e:  # show the problem under the controls instead of a long traceback
        status.value = f"<span style='color:#e8605a'>Could not show this file: {e}</span>"
        return
    colour_tb.disabled = truth is None
    if truth is None:
        colour_tb.value = "labels"
    # start at the ground truth when there is one, else at the first prediction
    start = state["rows"][0]["labels"]
    voxels = np.argwhere(start > 0)
    state.update(scan=scan, pos=[int(c) for c in voxels[np.abs(voxels - voxels.mean(0)).sum(1).argmin()]]
                 if len(voxels) else [n // 2 for n in scan.data.shape])
    for (_, axis, _, _), slider in zip(VIEWS, sliders):
        slider.max = scan.data.shape[axis] - 1
        slider.value = state["pos"][axis]
        slider.disabled = False
    grid.children = rows
    shape, spacing = scan.data.shape, scan.spacing
    status.value = (f"{Path(image_dd.value).name}: {shape[0]} x {shape[1]} x {shape[2]} voxels, "
                    f"{spacing[0]:.2g} x {spacing[1]:.2g} x {spacing[2]:.2g} mm")
    state["ready"] = True
    render()


def on_filter(change):
    terms = change["new"].lower().split()
    image_dd.options = [f for f in files if all(t in f.lower() for t in terms)]
    on_image({"new": image_dd.value})


def on_image(change):
    if not change["new"]:
        return
    guess = guess_mask(change["new"])
    if guess not in label_dd.options:
        label_dd.options = list(label_dd.options) + [guess]
    label_dd.value = guess


search.observe(on_filter, names="value")
image_dd.observe(on_image, names="value")
go_btn.on_click(on_go)
for control in (colour_tb, contrast, opacity, crosshair, size):
    control.observe(render, names="value")
for i, slider in enumerate(sliders):
    slider.observe(lambda change, i=i: on_slide(i, change), names="value")

on_image({"new": image_dd.value})
display(w.VBox([search, image_dd, label_dd, runs_sel, w.HBox([go_btn, status]),
                w.HBox([colour_tb, legend]), w.HBox([contrast, opacity, crosshair, size]),
                w.HBox(sliders), positions, grid, info]))